# Using ChromBERT-tools with a Apptainer Container

This notebook demonstrates how to use ChromBERT-tools commands with a Apptainer container.

## Key Apptainer Parameters

- `--nv`: Enable NVIDIA GPU support (required for GPU acceleration)

- `--bind`: Mount local directories into the container (format: `--bind /local/path:/container/path`)

- `--pwd`: Set working directory inside the container

## Notes

- All `chrombert-tools` commands work the same way inside the container.

- Running chromBERT-tools in the container produces the same outputs (format and directory structure) as running it on the host after a normal installation.

- For detailed command usage and output analysis, refer to other tutorial notebooks (e.g., `embed_regulator.ipynb`, `predict_cell_type_master_regulators.ipynb`).

- You don’t need to launch Jupyter from the container image.

In [1]:
import os
workdir="/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli" # your workdir
os.chdir(workdir)
os.environ["CUDA_VISIBLE_DEVICES"] = "2" # gpu device

In [2]:
sif_file = "/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/Singularity/chrombert-tools.sif" # your image file


! apptainer exec --nv {sif_file} chrombert-tools -h

INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
INFO:    gocryptfs not found, will not be able to use gocryptfs


Usage: chrombert-tools [OPTIONS] COMMAND [ARGS]...

  Type -h or --help after any subcommand for more information.

Options:
  -v, --verbose  Verbose logging
  -d, --debug    Post mortem debugging
  -V, --version  Show the version and exit.
  -h, --help     Show this message and exit.

Commands:
  embed_region                    Generate region...
  embed_regulator                 Extract regulator...
  gene_activity_regression        Predict gene...
  interpret_region_region_interactions
                                  Region embedding...
  interpret_regulator_effects_between_region_groups
                                  Identify regulators...
  interpret_regulator_regulator_interactions
                                  Interpret...
  predict_cell_type_master_regulators
                                  Find candidate key...
  predict_regulator_context_cofactors
                                  Identify...
  predict_tf_binding_regions      Predict TF binding...
  predict_transit

In [3]:
# Define example data file
region_file = '../data/CTCF_ENCFF664UGR_sample100.bed'


## Basic Usage: Check Available Commands


In [4]:
! apptainer exec {sif_file} chrombert-tools embed_regulator -h

INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
INFO:    gocryptfs not found, will not be able to use gocryptfs


Usage: chrombert-tools embed_regulator [OPTIONS]

  Extract regulator embeddings on specified regions. Supports both general and
  cell-specific modes.

Options:
  --region FILE                   Region file.  [required]
  --regulator TEXT                Regulators of interest, e.g. EZH2 or
                                  EZH2;BRD4. Use ';' to separate multiple
                                  regulators.  [required]
  --cell-type-bw FILE             Cell type accessibility BigWig file. Used
                                  for cell-specific mode.
  --cell-type-peak FILE           Cell type accessibility Peak BED file. Used
                                  for cell-specific mode.
  --ft-ckpt FILE                  Fine-tuned checkpoint. If provided, use
                                  cell-specific model and skip fine-tuning.
  --odir DIRECTORY                Output directory.  [default: ./output]
  --oname TEXT                    Output name of the regulator embeddings.
        

## Example 1: Extract Regulator Embeddings

This example demonstrates running `embed_regulator` with all necessary apptainer parameters.


In [5]:
# Run embed_regulator command inside apptainer container
# --nv: Enable NVIDIA GPU
# --bind: Mount local directory to container
# --pwd: Set working directory inside container
! apptainer exec --nv \
    --bind /mnt/Storage2/home/chenqianqian/:/mnt/Storage2/home/chenqianqian/ \
    --pwd {workdir} \
    {sif_file} \
    chrombert-tools embed_regulator \
    --region {region_file} \
    --regulator "EZH2;BRD4;CTCF;FOXA3;myod1;myF5" \
    --odir "./output_emb_regulator_appatiner" \
    --genome "hg38" \
    --resolution "1kb"

INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
INFO:    gocryptfs not found, will not be able to use gocryptfs
Region summary - total: 100, overlapping with ChromBERT: 100 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Note: All regulator names were converted to lowercase for matching.
Regulator count summary - requested: 6, matched in ChromBERT: 5, not found: 1, not found regulator: ['foxa3']
ChromBERT regulators: /mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_regulators_list.txt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
Your supervised_file does not contain the 'label' column. 

## Example 2: Infer regulator-regulator interaction networks

This example demonstrates running `interpret_regulator_regulator_interactions` with all necessary apptainer parameters.


In [6]:
! apptainer exec --nv \
    --bind /mnt/Storage2/home/chenqianqian/:/mnt/Storage2/home/chenqianqian/ \
    --pwd {workdir} \
    {sif_file} \
    chrombert-tools interpret_regulator_regulator_interactions \
    --region "../data/CTCF_ENCFF664UGR_sample100.bed" \
    --regulator "ctcf" \
    --odir "./output_trn_apptainer_1kb" \
    --genome "hg38" \
    --resolution "1kb"

INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
INFO:    gocryptfs not found, will not be able to use gocryptfs
Region summary - total: 100, overlapping with ChromBERT: 100 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Note: All regulator names were converted to lowercase for matching.
Regulator count summary - requested: 1, matched in ChromBERT: 1, not found: 0, not found regulator: []
ChromBERT regulators: /mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_regulators_list.txt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
100%|█████████████████████████████████████████████| 2/2 [00:03<00

## Example 3: Predict TF binding regions

This example demonstrates running `predict_tf_binding_regions` with all necessary apptainer parameters.

In [7]:
! apptainer exec --nv \
    --bind /mnt/Storage2/home/chenqianqian/:/mnt/Storage2/home/chenqianqian/ \
    --pwd {workdir} \
    {sif_file} \
    chrombert-tools predict_tf_binding_regions \
    --cistrome "BCL11A:GM12878;BRD4:MCF7;CTCF:HepG2;MYC:H1;MYC:h9;SPI1:GSM2702714" \
    --region "../data/CTCF_ENCFF664UGR_sample100.bed" \
    --odir "./output_predict_tf_binding_regions_apptainer" \
    --genome "hg38" \
    --resolution "1kb"

INFO:    fuse2fs not found, will not be able to mount EXT3 filesystems
INFO:    gocryptfs not found, will not be able to use gocryptfs
Region summary - total: 100, overlapping with ChromBERT: 100 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
celltype: h1 has no corresponding wild type dnase data in ChromBERT.
Note: All cistromes names were converted to lowercase for matching.
Cistromes count summary - requested: 6, matched in ChromBERT: 5, not found: 1, not found cistromes: ['myc:h1']
ChromBERT cistromes metas: /mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_meta.tsv
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is requir

## Example 4: Infer cell-type-specific key regulators

This example demonstrates running `predict_cell_type_master_regulators` with all necessary apptainer parameters.

In [ ]:
! apptainer exec --nv \
    --bind /mnt/Storage2/home/chenqianqian/:/mnt/Storage2/home/chenqianqian/ \
    --pwd {workdir} \
    {sif_file} \
    chrombert-tools predict_cell_type_master_regulators \
    --cell-type-bw "../data/myoblast_ENCFF149ERN_signal.bigwig" \
    --cell-type-peak "../data/myoblast_ENCFF647RNC_peak.bed" \
    --odir "./output_predict_cell_type_master_regulators_apptainer" \
    --genome "hg38" \
    --resolution "1kb"  2> "./tmp/infer_cell_key_regulator.sif.stderr.log" # redirect stderr to log file

Step 1/3: Building or loading a cell-specific model...
Preparing dataset ...
Region summary - total: 373422, overlapping with ChromBERT: 368260 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 7920
Total regions: 324690
Fast mode: downsampling to 20k regions
Fine-tuning cell-specific model...

[Attempt 0/2] seed=55
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Epoch 0:  20%|████▍                 | 800/4000 [02:18<09:15,  5.77it/s, v_num=0]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|████████████████| 250/250 [00:24<00:00, 10.20it/s]
Epoch 0:  40%|▍| 1600/4000 [05:01<07:32,  5.30it/s, v_num=0, default_validation/
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|████████████████|

## Analyzing Output Files

The output files generated using Singularity are identical to those from direct command-line execution - both methods produce the same results., refer to other tutorial notebooks